# Lab 18 extra: measure the designs over several runs

**Going further, after notebooks 1 and 2**

## Goal

Notebooks 1 and 2 ran each design once and read that one run. A model writes a new reply on every run, so one run
cannot say whether the answer or the cost stays the same. This notebook runs one agent and the parallel sub-agents five times
each, on the book club's question and on the desk's five customers, and puts every run in one table. It then tests,
with no model at all, why the desk's lookups each open their own database connection. It ends with an A2A agent card:
how an agent that another team runs describes itself to other agents.

## What you will learn

- How to record one row per run (the answer checked, requests, input tokens, the answering request) and read a
  design's spread over five runs.
- How to show that a SQLite connection shared by tools running at the same time returns wrong results, and that a
  connection per lookup does not.
- What an A2A agent card contains, which of its fields the A2A specification requires, and how it compares with an MCP
  (Model Context Protocol) tool listing from Lab 06.

## Setup

The same configuration cell as every lab, then the two settings every agent gets.

In [1]:
from pathlib import Path
import asyncio
import json
import os
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import AsyncOpenAI
from openai.types.shared import Reasoning
from agents import Agent, ModelSettings, Runner, function_tool, set_default_openai_client

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))
SETTINGS = ModelSettings(reasoning=Reasoning(effort=EFFORT))

The agents of notebooks 1 and 2, restated so this notebook runs on its own. Nothing here is new, so the cells are
long; notebooks 1 and 2 build each piece one at a time. First the book club: the wishes, the blurbs, `read_blurb`, the
one agent (`helper`), the reader and the chooser.

In [3]:
CLUB = ("Our book club meets in three weeks. What we want from the next book: no more than 350 pages; short "
        "chapters, because most of us read on the bus; a happy ending, because last month's book made half of us "
        "cry; and a book that stands on its own, because we have not read anything else by these authors.")

BOOKS = {}
for path in sorted((ROOT / "18_Agent_Designs" / "book_club").glob("*.md")):
    text = path.read_text(encoding="utf-8")
    BOOKS[text.splitlines()[0].removeprefix("# ")] = text


def read_blurb(title: str) -> str:
    """Read the publisher's blurb of one book on the club's shortlist.

    Args:
        title: The book's title, exactly as it is on the shortlist.
    """
    for known_title, text in BOOKS.items():
        if known_title.lower() == title.strip().lower():
            return text
    return f"There is no book called {title!r} on the shortlist."


READ_BLURB = function_tool(read_blurb)
QUESTION = "Which of these five books should our book club read next? " + "; ".join(BOOKS) + "."

helper = Agent(name="Book club helper", model=MODEL, tools=[READ_BLURB], model_settings=SETTINGS,
               instructions=("You help a book club choose its next book. Read the blurbs with your tool before you "
                             "answer. " + CLUB + " Answer in a few sentences."))
reader = Agent(name="Blurb reader", model=MODEL, tools=[READ_BLURB], model_settings=SETTINGS,
               instructions=("You read the blurb of one book for a book club. " + CLUB + " Read the blurb with your "
                             "tool, then reply with one line: the title, whether it fits, and why."))
chooser = Agent(name="Book club chooser", model=MODEL, model_settings=SETTINGS,
                instructions=("You help a book club choose its next book. " + CLUB + " Answer from the notes you are "
                              "given. Answer in a few sentences."))

Then the desk's three lookups, each on a database connection of its own (Part B shows why):

In [4]:
from contextlib import closing

DB_URI = f"file:{ROOT / 'data' / 'chinook.db'}?mode=ro"


def query(sql, args):
    """Run one read-only query on a connection of its own, and return every row."""
    with closing(sqlite3.connect(DB_URI, uri=True)) as conn:
        return conn.execute(sql, args).fetchall()


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = query("SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
                 "WHERE FirstName || ' ' || LastName = ?", (name,))
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = query("SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,))[0]
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    rows = query("SELECT e.FirstName || ' ' || e.LastName FROM customers c "
                 "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,))
    return {"customer_id": customer_id, "support_rep": rows[0][0] if rows else None}

And the desk's request and its three agents: the desk agent with every lookup, the worker for one customer, and the
main agent with no tools.

In [5]:
FIVE = ["Helena Holý", "Puja Srivastava", "Richard Cunningham", "Luis Rojas", "Manoj Pareek"]
TOTALS = ("What has each of these five customers spent in total: Helena Holý, Puja Srivastava, Richard Cunningham, "
          "Luis Rojas and Manoj Pareek? Who spent the most?")

desk = Agent(name="Desk", model=MODEL, model_settings=SETTINGS,
             tools=[function_tool(find_customer), function_tool(get_invoices), function_tool(get_support_rep)],
             instructions=("You help the support reps of a music shop. Look up every fact with the tools; "
                           "never answer from memory. Answer in one or two sentences."))
worker = Agent(name="Customer worker", model=MODEL, model_settings=SETTINGS,
               tools=[function_tool(find_customer), function_tool(get_invoices)],
               instructions=("You look up one customer's total spend. Look up every fact with your tools; never "
                             "answer from memory. Reply with one line: the name and the total in dollars."))
main = Agent(name="Main", model=MODEL, model_settings=SETTINGS,
             instructions="You help the support reps of a music shop. Answer from the lines you are given; do not add facts.")
print("agents:", [agent.name for agent in [helper, reader, chooser, desk, worker, main]])

agents: ['Book club helper', 'Blurb reader', 'Book club chooser', 'Desk', 'Customer worker', 'Main']


---

## Part A: Record one row per run

Every run of a design becomes one dictionary with the same keys, so the runs can be printed as a table and compared:

- `design` and `run`: which design, and which of its five runs;
- `right`: whether the answer passed the check for that question (written below for each question);
- `requests` and `input`: model requests and input tokens, added up over every agent run the design made;
- `answering`: the input tokens of the answering request, the last request of the agent that wrote the answer.

`record` builds that row. `runs` is the list of the design's `RunResult`s (one for one agent; the readers or workers
and the agent that answered, for the split), and `answering` is the run that wrote the answer.

In [6]:
ROWS = []


def record(design, run_number, right, runs, answering):
    ROWS.append({
        "design": design,
        "run": run_number,
        "right": right,
        "requests": sum(run.context_wrapper.usage.requests for run in runs),
        "input": sum(run.context_wrapper.usage.input_tokens for run in runs),
        "answering": answering.raw_responses[-1].usage.input_tokens,
    })
    print(ROWS[-1])

### The book club's question, five runs of each design

The check: the answer names *The Lighthouse Bakery*, the only book that meets all four wishes. It looks only for the
title in the text, so an answer that names the book and turns it down would pass as well. The split is the code
of notebook 1, Part C, in one cell.

In [7]:
for number in range(1, 6):
    alone = await Runner.run(helper, QUESTION)
    record("book club, one agent", number, "Lighthouse Bakery" in alone.final_output, [alone], alone)

for number in range(1, 6):
    reader_runs = await asyncio.gather(
        *[Runner.run(reader, f"Read '{title}' and tell us whether it fits.") for title in BOOKS])
    notes = "\n".join(run.final_output for run in reader_runs)
    chosen = await Runner.run(chooser, QUESTION + "\n\nNotes from the readers:\n" + notes)
    record("book club, five readers", number, "Lighthouse Bakery" in chosen.final_output, [*reader_runs, chosen], chosen)

{'design': 'book club, one agent', 'run': 1, 'right': True, 'requests': 2, 'input': 2295, 'answering': 2083}


{'design': 'book club, one agent', 'run': 2, 'right': True, 'requests': 2, 'input': 2295, 'answering': 2083}


{'design': 'book club, one agent', 'run': 3, 'right': True, 'requests': 2, 'input': 2295, 'answering': 2083}


{'design': 'book club, one agent', 'run': 4, 'right': True, 'requests': 2, 'input': 2295, 'answering': 2083}


{'design': 'book club, one agent', 'run': 5, 'right': True, 'requests': 2, 'input': 2295, 'answering': 2083}


{'design': 'book club, five readers', 'run': 1, 'right': True, 'requests': 11, 'input': 4197, 'answering': 328}


{'design': 'book club, five readers', 'run': 2, 'right': True, 'requests': 11, 'input': 4196, 'answering': 327}


{'design': 'book club, five readers', 'run': 3, 'right': True, 'requests': 11, 'input': 4191, 'answering': 322}


{'design': 'book club, five readers', 'run': 4, 'right': True, 'requests': 11, 'input': 4222, 'answering': 353}


{'design': 'book club, five readers', 'run': 5, 'right': True, 'requests': 11, 'input': 4221, 'answering': 352}


### The desk's five customers, five runs of each design

The check: the answer contains all five totals from the database. It does not check which customer the answer names as the top spender.

In [8]:
RIGHT_TOTALS = ["49.62", "36.64", "47.62", "46.62", "38.62"]


def has_all_totals(answer):
    return all(total in answer for total in RIGHT_TOTALS)


for number in range(1, 6):
    alone = await Runner.run(desk, TOTALS)
    record("desk, one agent", number, has_all_totals(alone.final_output), [alone], alone)

for number in range(1, 6):
    worker_runs = await asyncio.gather(*[Runner.run(worker, f"What has {name} spent in total?") for name in FIVE])
    lines = "\n".join(run.final_output for run in worker_runs)
    joined = await Runner.run(main, TOTALS + "\n\nWhat the workers found:\n" + lines)
    record("desk, five workers", number, has_all_totals(joined.final_output), [*worker_runs, joined], joined)

{'design': 'desk, one agent', 'run': 1, 'right': True, 'requests': 3, 'input': 1472, 'answering': 756}


{'design': 'desk, one agent', 'run': 2, 'right': True, 'requests': 3, 'input': 1472, 'answering': 756}


{'design': 'desk, one agent', 'run': 3, 'right': True, 'requests': 3, 'input': 1472, 'answering': 756}


{'design': 'desk, one agent', 'run': 4, 'right': True, 'requests': 3, 'input': 1472, 'answering': 756}


{'design': 'desk, one agent', 'run': 5, 'right': True, 'requests': 3, 'input': 1472, 'answering': 756}


{'design': 'desk, five workers', 'run': 1, 'right': True, 'requests': 16, 'input': 3364, 'answering': 125}


{'design': 'desk, five workers', 'run': 2, 'right': True, 'requests': 16, 'input': 3364, 'answering': 125}


{'design': 'desk, five workers', 'run': 3, 'right': True, 'requests': 16, 'input': 3364, 'answering': 125}


{'design': 'desk, five workers', 'run': 4, 'right': True, 'requests': 16, 'input': 3364, 'answering': 125}


{'design': 'desk, five workers', 'run': 5, 'right': True, 'requests': 16, 'input': 3364, 'answering': 125}


### All twenty runs in one table

One line per design: how many runs passed the check, and the lowest and highest value of each count over the five
runs (one value when all five agreed). `dict.fromkeys(...)` keeps each design name once, in the order the rows were
added, so the loop prints one line per design.

In [9]:
def low_high(values):
    return str(min(values)) if min(values) == max(values) else f"{min(values)}-{max(values)}"


print(f"{'design':26} {'right':>5} {'requests':>9} {'input tokens':>13} {'answering request':>18}")
for design in dict.fromkeys(row["design"] for row in ROWS):
    rows = [row for row in ROWS if row["design"] == design]
    right = f"{sum(row['right'] for row in rows)}/{len(rows)}"
    print(f"{design:26} {right:>5} {low_high([r['requests'] for r in rows]):>9} "
          f"{low_high([r['input'] for r in rows]):>13} {low_high([r['answering'] for r in rows]):>18}")

design                     right  requests  input tokens  answering request
book club, one agent         5/5         2          2295               2083
book club, five readers      5/5        11     4191-4222            322-353
desk, one agent              5/5         3          1472                756
desk, five workers           5/5        16          3364                125


Read it one question at a time:

- **The book club:** both designs named *The Lighthouse Bakery* in all five runs. One agent made the same 2 requests every time, and its answering request read 2,083 tokens every time. The five readers took 11 requests in every run; their lines changed a little in wording, so the chooser's answering request moved between 322 and 353 tokens, and the total between 4,191 and 4,222.
- **The desk:** both designs gave all five totals in all five runs, at fixed counts: one agent 3 requests, 1,472 input tokens and an answering request of 756; the five workers 16 requests, 3,364 tokens and an answering request of 125.

So the single run each notebook printed was a typical one: on this model, at this size, the counts hardly move. They change when the work changes: more books, longer blurbs or more customers (notebook 2's Challenge grows the request to ten).

**Checkpoint A.** Twenty rows and a four-line table. Every design passed its check in 5 of 5 runs; with the split, the answering request about a sixth of one agent's (322 to 353 against 2,083 for the book club, 125 against 756 at the desk); the split's requests 11 and 16 against 2 and 3.

---

## Part B: Test whether one database connection is safe for tools that run at the same time

Notebook 2's lookups each open their own connection. The reason is in this part, with no model involved. When the
model asks for several lookups in one reply, the Agents SDK runs each tool function in a thread of its own, at the
same time (`asyncio.to_thread`). Here we do the same thing ourselves: five `find_customer` calls at the same time,
200 times, once with one connection shared by every call and once with the `find_customer` of notebook 2.

The shared version is the usual way to open a database once in a notebook: one connection, and
`check_same_thread=False` so that other threads may use it.

In [10]:
from collections import Counter

shared = sqlite3.connect(DB_URI, uri=True, check_same_thread=False)


def find_customer_shared(name):
    rows = shared.execute("SELECT FirstName || ' ' || LastName FROM customers "
                          "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"name": r[0]} for r in rows]


async def one_lookup(function, name):
    """Run one lookup in a thread, as the SDK does, and say what came back."""
    try:
        found = await asyncio.to_thread(function, name)
    except sqlite3.Error as error:
        return f"{type(error).__name__}: {error}"
    if not found:
        return "empty"
    return "right" if found[0]["name"] == name else "another customer's row"


for function in [find_customer_shared, find_customer]:
    outcomes = Counter()
    for _ in range(200):
        outcomes.update(await asyncio.gather(*[one_lookup(function, name) for name in FIVE]))
    print(function.__name__, dict(outcomes))

find_customer_shared {'right': 888, 'empty': 50, 'InterfaceError: bad parameter or other API misuse': 44, "another customer's row": 18}


find_customer {'right': 1000}


With one shared connection, 112 of the 1,000 lookups went wrong: 50 came back empty, 44 raised `InterfaceError: bad parameter or other API misuse`, and 18 returned another customer's row. With a connection per lookup, all 1,000 were right. The wrong counts change from run to run, because they depend on how the threads happen to overlap.

An error stops the lookup, and you see it in the output. The other two failures give no sign: an empty list reads as "no such customer", another customer's row reads as a fact, and an agent repeats both in a fluent reply. A model that asks for five lookups in one reply makes these overlaps happen on every such turn.

**Key lesson:** a tool that runs at the same time as others must be safe to run that way. A model that asks for five
lookups in one reply is doing what we want; the tool code decides whether the five results are right.

**Checkpoint B.** The shared connection gives some empty results, some rows of another customer, and some
`InterfaceError` lines out of 1,000 lookups; `find_customer` with its own connection gives 1,000 right.

---

## Part C: Describe an agent to other agents with an A2A agent card

Every agent in this lab ran in our notebook, so our code could call it directly. When the second agent belongs to
another team and runs on their server, our code needs to find out what it does and how to reach it. **A2A**
(Agent2Agent) is an open protocol for that: one agent sends another a task over HTTP, and the other agent does the
work with its own code, tools and model, and returns the result. The specification's latest released version is
1.0.0.

An A2A agent publishes an **agent card**, a JSON document at `https://<its domain>/.well-known/agent-card.json`. A
client reads the card, picks the first interface it supports, and sends its message there (in the JSON-RPC binding,
with the method `SendMessage`). Here is a card for the desk's worker agent, as another team would see it if we offered
the worker as an A2A agent. It is a plain dictionary; nothing is served or sent.

In [11]:
card = {
    "name": "Chinook customer totals",
    "description": "Looks up one customer of the Chinook music shop and replies with their total spend in one line.",
    "supportedInterfaces": [
        {"url": "https://desk.chinook.example/a2a", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"},
    ],
    "version": "1.0.0",
    "capabilities": {"streaming": False, "pushNotifications": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [
        {
            "id": "customer-total",
            "name": "Customer total spend",
            "description": "Finds a customer by full name and adds up the totals of all their invoices, in US dollars.",
            "tags": ["customers", "invoices", "billing"],
            "examples": ["What has Helena Holý spent in total?"],
        }
    ],
}
print(json.dumps(card, indent=2, ensure_ascii=False))

{
  "name": "Chinook customer totals",
  "description": "Looks up one customer of the Chinook music shop and replies with their total spend in one line.",
  "supportedInterfaces": [
    {
      "url": "https://desk.chinook.example/a2a",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": false,
    "pushNotifications": false
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "customer-total",
      "name": "Customer total spend",
      "description": "Finds a customer by full name and adds up the totals of all their invoices, in US dollars.",
      "tags": [
        "customers",
        "invoices",
        "billing"
      ],
      "examples": [
        "What has Helena Holý spent in total?"
      ]
    }
  ]
}


The fields, and where each comes from in the worker we built:

| Field | What it says | In our worker |
|---|---|---|
| `name`, `description` | what the agent is, in words for people and for other agents | the agent's name and a sentence about its instructions |
| `supportedInterfaces` | where to send a message, over which binding (`JSONRPC`, `GRPC` or `HTTP+JSON`) and which protocol version; the first entry is the preferred one | nothing yet: the worker runs only inside our notebook (the address is made up) |
| `version` | the version of this agent, not of the protocol | our choice |
| `capabilities` | optional features such as streaming and push notifications | none |
| `defaultInputModes`, `defaultOutputModes` | media types the agent reads and writes | plain text in, one line of text out |
| `skills` | what the agent can be asked to do, each with an `id`, `name`, `description`, `tags` and optional `examples` | its one job, a customer's total spend |

The specification (section 4.4.1) lists eight required fields. A short check that the card has them all:

In [12]:
REQUIRED = ["name", "description", "supportedInterfaces", "version", "capabilities",
            "defaultInputModes", "defaultOutputModes", "skills"]
SKILL_REQUIRED = ["id", "name", "description", "tags"]

print("missing card fields: ", [field for field in REQUIRED if field not in card])
print("missing skill fields:", [field for skill in card["skills"] for field in SKILL_REQUIRED if field not in skill])

missing card fields:  []
missing skill fields: []


Both lists are empty: the card has every required field. This checks only that the keys are present; it says nothing
about whether an agent answers at that address.

**A2A and MCP.** An MCP tool listing (Lab 06) describes a **tool**: a name, a description and the JSON schema of its
arguments; you call it with exact arguments and it returns a result. An A2A card describes an **agent**: it lists
the agent's skills in words. You send the agent a task as a message, and it does the task with its own tools and
model, which you cannot see. The two fit together: an agent reached
over A2A may itself use MCP for its own tools.

**What we did not do.** Serving the card and answering `SendMessage` needs an A2A server, which the official SDKs
provide (Python among them); this repository does not install one. Inside one program, as in notebooks 1 and 2,
`Runner.run` or `as_tool` connects two agents with no protocol at all. A2A matters when the second agent belongs to
someone else.

**Checkpoint C.** The card prints as JSON with `supportedInterfaces` (binding `JSONRPC`, protocol version `1.0`) and
one skill, and both lists of missing fields are empty.

## What you can do now

- Record one row per run and read a design's spread over several runs (Part A).
- Show that one database connection shared by tools that run at the same time returns wrong results, and use a
  connection per lookup instead (Part B).
- Read an A2A agent card, check its required fields, and say how it differs from an MCP tool listing (Part C).

## Resources

- [A2A specification](https://a2a-protocol.org/latest/specification/): the agent card (section 4.4), discovery at
  `/.well-known/agent-card.json` (section 8.2), and the relationship to MCP (Appendix B)
- [A2A and MCP](https://a2a-protocol.org/latest/topics/a2a-and-mcp/): how the two protocols fit together
- [OpenAI Agents SDK: usage](https://openai.github.io/openai-agents-python/usage/): requests and tokens per run
- [Python `sqlite3`](https://docs.python.org/3/library/sqlite3.html): connections, `check_same_thread`, and how the
  module behaves with threads
- [Python `asyncio.to_thread`](https://docs.python.org/3/library/asyncio-task.html#running-in-threads): running a
  plain function in a thread from async code

Back to [notebook 1](18a_meet_agent_designs.ipynb) or [notebook 2](18b_agent_designs_at_the_desk.ipynb).